# Watershed delineation from DEM tiles

This notebook mosaics DEM tiles, fills depressions, computes D8 flow direction and flow
accumulation, delineates the catchment above a pour point, and samples elevation along
flowlines. It writes a QGIS bundle (GeoTIFFs, a GeoPackage, QML styles, a loader script) and a
Kepler.gl map under `exports/hydrology/`.

The DEM is synthetic. `datasets.synthetic.synthetic_dem` builds a 240 by 300 cell surface from
six Gaussian hills plus noise, in UTM zone 15N at 100 m resolution, and
`synthetic_dem_tiles` writes it as four GeoTIFF tiles under `data/hydrology/`. Point `tiles` at
real tiles and `pour_point` at a gauge and the rest of the notebook runs unchanged.

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
warnings.filterwarnings("ignore")

DATA = ROOT / "data" / "hydrology"
EXPORTS = ROOT / "exports" / "hydrology"
DATA.mkdir(parents=True, exist_ok=True)
EXPORTS.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("repo root:", ROOT)

repo root: /mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects


In [2]:
from datasets.synthetic import synthetic_dem, synthetic_dem_tiles
from fortress_gis.domains import hydrology as hy
from fortress_gis.stats.regression import fit_ols
from fortress_gis.viz.kepler import KeplerMapBuilder, kepler_available

tiles = sorted(DATA.glob("dem_tile_*.tif"))
if not tiles:
    tiles = synthetic_dem_tiles(synthetic_dem(240, 300), DATA)
tiles

[PosixPath('/mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects/data/hydrology/dem_tile_r0_c0.tif'),
 PosixPath('/mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects/data/hydrology/dem_tile_r0_c1.tif'),
 PosixPath('/mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects/data/hydrology/dem_tile_r1_c0.tif'),
 PosixPath('/mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects/data/hydrology/dem_tile_r1_c1.tif')]

## Mosaic the tiles

`load_dem` merges any number of tiles with `rasterio.merge` and returns a `RasterInfo`, a small
record holding the array, transform, CRS and nodata value. Every terrain function in
`fortress_gis.raster` takes and returns the same record, so intermediate grids can be written
with `write_raster` at any step.

In [3]:
dem = hy.load_dem(tiles)
print("shape", dem.shape, "resolution", dem.resolution, "crs", dem.crs)
print("bounds", dem.bounds)

fig, ax = plt.subplots(figsize=(7, 5))
b = dem.bounds
extent = (b[0], b[2], b[1], b[3])
im = ax.imshow(dem.masked(), cmap="terrain", extent=extent)
fig.colorbar(im, ax=ax, label="elevation (m)")
ax.set_title("Synthetic DEM (4 tiles merged)")
plt.show()

shape (240, 300) resolution (100.0, 100.0) crs EPSG:32615
bounds (500000.0, 4400000.0, 530000.0, 4424000.0)


## Pick a pour point

The pour point is the lowest cell on the western edge, which is where the synthetic surface
drains. With a real DEM this would be a gauge or culvert location, given either as an `(x, y)`
tuple in the DEM CRS or as a one-row GeoDataFrame in any CRS. `delineate_watershed` snaps the
point to the highest accumulation cell within `snap_cells` (default 5), which absorbs the usual
offset between a gauge coordinate and the raster channel.

In [4]:
elev = dem.masked()
xs, ys = dem.cell_centers()
west = np.where(np.isfinite(elev[:, 0]), elev[:, 0], np.inf)
row = int(np.argmin(west))
pour_point = (float(xs[row, 0]), float(ys[row, 0]))
pour_point

(500050.0, 4412050.0)

## Delineate the catchment

`run_watershed_pipeline` chains the steps: fill depressions (priority flood with a small
epsilon so flats drain), D8 flow direction, flow accumulation, snap and trace the catchment,
threshold accumulation into a stream mask, and trace flowlines down from stream heads. The
stream threshold defaults to 1 percent of the grid, here 720 cells.

In [5]:
result = hy.run_watershed_pipeline(dem, pour_point, n_synthetic_flowlines=14)
result.summary().round(2)

dem_cells                    72000.00
catchment_cells              24154.00
catchment_area_km2             241.54
outlet_accumulation_cells    24154.00
stream_threshold_cells         720.00
stream_cells                  1793.00
min_elevation                  139.73
max_elevation                  866.98
catchment_elev_mean            442.42
catchment_elev_min             140.41
catchment_elev_max             866.10
dtype: float64

In [6]:
fig, ax = plt.subplots(figsize=(8, 6))
acc = result.accumulation.masked()
im = ax.imshow(np.log10(acc + 1), cmap="Blues", extent=extent)
fig.colorbar(im, ax=ax, label="log10(accumulation cells + 1)")
result.catchment_polygon.boundary.plot(ax=ax, color="red", linewidth=1.5)
result.flowlines.plot(ax=ax, color="black", linewidth=0.8)
result.outlet_point.plot(ax=ax, color="orange", markersize=60, zorder=5)
ax.set_title("Flow accumulation, catchment boundary (red), flowlines, outlet")
plt.show()

## Elevation along flowlines

`attach_flowline_elevations` densifies each line to one vertex per cell, samples the filled DEM
at every vertex, and summarises per line: mean, min and max elevation, length in km, and slope
in metres per km from the drop between endpoints.

In [7]:
lines = result.flowlines.drop(columns="geometry")
lines.sort_values("length_km", ascending=False).round(2).head(10)

,line_id,elevation_mean,elevation_min,elevation_max,length_km,slope_m_per_km
11,12,282.57,140.41,506.16,20.43,17.90
10,11,270.17,140.41,489.46,19.20,18.18
0,1,259.99,140.41,480.15,18.40,18.47
12,13,251.77,140.41,392.46,16.17,15.59
2,3,255.60,140.41,386.10,16.07,15.29
4,5,245.40,140.41,365.88,14.82,15.22
9,10,237.09,140.41,372.66,14.39,16.14
3,4,228.41,140.41,353.02,13.53,15.72
6,7,225.88,140.41,352.86,13.13,16.18
13,14,224.08,140.41,346.60,13.04,15.81


## Slope against channel length

Channel slope usually falls as drainage length grows: headwater reaches are steep and the
mainstem is flat. The regression below checks that on the traced lines. With a dozen lines the
fit is only indicative; the point is the call pattern, which is the same for any table.
`fit_ols` uses HC3 standard errors by default.

In [8]:
lines["log_length_km"] = np.log(lines["length_km"])
ols = fit_ols(lines, target="slope_m_per_km", features=["log_length_km"])
print(ols.metrics())
ols.coefficients().set_index("term").round(3)

kind                ols
n_obs                14
n_features            1
r2             0.832014
r2_adj         0.818015
aic           65.613247
bic           66.891362
dtype: object


,estimate,std_error,statistic,p_value,ci_low,ci_high
term,,,,,,
const,38.591,3.802,10.151,0.0,31.140,46.042
log_length_km,-8.081,1.505,-5.370,0.0,-11.031,-5.132


## Kepler.gl map

The cell below renders the map inline. If the widget shows as blank text, run
`jupyter nbextension enable --py --sys-prefix keplergl` once in the environment and reload the
page; `fortress_gis.viz.kepler.enable_nbextension()` prints the same command. The HTML export in
the last section does not need the extension and opens in any browser.

In [9]:
builder = KeplerMapBuilder(title="Watershed", height=550)
builder.add_layer(result.catchment_polygon, "catchment", opacity=0.25)
builder.add_layer(
    result.flowlines,
    "flowlines",
    color_field="slope_m_per_km",
    colors=("#ffffcc", "#fd8d3c", "#800026"),
)
builder.add_layer(result.outlet_point, "outlet", radius=8)
builder.widget() if kepler_available() else print("keplergl not installed")

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'visState': {'layers': [{'id': 'catchment', 'type': 'geojson', 'c…

## Export for QGIS and the browser

`export_artifacts` writes the DEM, filled DEM, accumulation and stream rasters as GeoTIFFs, the
catchment, flowlines and outlet into one GeoPackage, a QML style per layer, a `manifest.json`,
and `load_in_qgis.py`. In QGIS, open the Python console and run
`exec(open("<path>/load_in_qgis.py").read())` to load everything styled. The Kepler HTML is a
single file that opens in any browser.

In [10]:
paths = hy.export_artifacts(result, EXPORTS, name="watershed")
for k, v in paths.items():
    print(f"{k:>10}: {v.relative_to(ROOT)}")

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter
Map saved to /mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects/exports/hydrology/watershed_kepler.html!
      qgis: exports/hydrology/qgis
kepler_html: exports/hydrology/watershed_kepler.html
kepler_config: exports/hydrology/watershed_kepler_config.json
